# Document Processing Pipeline
**Stack:** YOLOv11s (DocLayNet) · PaddleOCR · Gemini (vision + text) · Python 3.10+

**Pipeline stages:**
1. Preprocessing — render PDF / normalize image
2. Layout detection — YOLO → bbox cleanup → reading order
3. OCR routing — Text/Title/Caption → PaddleOCR | Table/Formula → Gemini Vision | low-conf → recovery
4. LLM post-processing — fix OCR, normalize Markdown / LaTeX / JSON
5. Schema validation + retry
6. Page summary → final structured output

## 0. Install dependencies

In [1]:
# # ── Run once — Python 3.11.9 venv · RTX 3050 Ti · CUDA 12.4 ───────────────────
# # Kernel to select: "Pipeline (Python 3.11.9 GPU)"
# # Venv path     : C:\Users\HP\Desktop\Kubbies\KhoaLuan\.venv311

# import sys
# print(f"Python : {sys.executable}")
# print(f"Version: {sys.version.split()[0]}")
# assert "3.11" in sys.version, "Wrong kernel! Select 'Pipeline (Python 3.11.9 GPU)' then rerun."

# PYTHON = sys.executable

# # 1. PyTorch + CUDA 12.4
# print("\n[1/7] PyTorch CUDA 12.4...")
# !"{PYTHON}" -m pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# print("  done.")

# # 2. YOLO
# print("[2/7] ultralytics...")
# !"{PYTHON}" -m pip install -q ultralytics
# print("  done.")

# # 3. PaddlePaddle GPU 3.0 (cu123 — compatible with driver 13.0)
# print("[3/7] PaddlePaddle GPU 3.0 (downloading ~800 MB — may take a few minutes)...")
# !"{PYTHON}" -m pip install -q paddlepaddle-gpu==3.0.0rc1 -i https://www.paddlepaddle.org.cn/packages/stable/cu123/
# print("  done.")

# # 4. PaddleOCR (3.x pulls paddlex → needs langchain deps)
# print("[4/7] PaddleOCR + langchain deps...")
# !"{PYTHON}" -m pip install -q paddleocr
# !"{PYTHON}" -m pip install -q langchain-text-splitters langchain-community
# print("  done.")

# # 5. Document / image processing
# print("[5/7] pdf2image / Pillow / opencv...")
# !"{PYTHON}" -m pip install -q pdf2image Pillow opencv-python-headless
# print("  done.")

# # 6. Gemini + utilities
# print("[6/7] Gemini + utilities...")
# !"{PYTHON}" -m pip install -q "google-generativeai<1.0" pydantic tqdm matplotlib numpy
# print("  done.")

# # ── Verify ─────────────────────────────────────────────────────────────────────
# print("\n[7/7] Verifying...")
# import torch
# print(f"PyTorch  : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
# print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
# print("\nAll dependencies installed.")

## 1. Configuration

In [ ]:
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import Literal

# ── BaseC profiling summary ────────────────────────────────────────────
# Model  : YOLOv11s, run_C_640_balanced, 20 epochs (Kaggle T4)
# Results: mAP50=0.9218  mAP50-95=0.7466  P=0.8805  R=0.8685
# Dataset: DocLayNet — all GT images are 1025×1025 px
# ──────────────────────────────────────────────────────────────────────

@dataclass
class Config:
    # ── Paths ──────────────────────────────────────────────────────────
    yolo_weights: str      = r"C:\Users\HP\Desktop\Kubbies\KhoaLuan\dataset\BaseC\best.pt"
    gemini_api_key: str    = os.getenv("GEMINI_API_KEY", "")
    output_dir: Path       = Path("output")

    # ── YOLO ───────────────────────────────────────────────────────────
    yolo_conf: float       = 0.20    # lowered: GT avg 15 blocks/page, 0.30 misses too many
    yolo_iou: float        = 0.60
    yolo_imgsz: int        = 640
    yolo_max_det: int      = 300     # GT max=46, headroom for dense pages
    yolo_device: str       = "0"

    # ── GT coordinate space ────────────────────────────────────────────
    gt_imgsz: int          = 1025    # DocLayNet GT is always 1025×1025

    # ── OCR routing ────────────────────────────────────────────────────
    ocr_conf_threshold: float = 0.70
    ocr_lang: list         = field(default_factory=lambda: ["en"])

    # ── PDF rendering ──────────────────────────────────────────────────
    pdf_dpi: int           = 150
    ocr_dpi: int           = 300
    ocr_min_crop_h: int    = 64
    poppler_path: str      = r"C:\Users\HP\Desktop\Kubbies\KhoaLuan\poppler\Library\bin"

    # ── Gemini ─────────────────────────────────────────────────────────
    gemini_model: str      = "gemini-2.5-flash"
    max_retries: int       = 3

    # ── DocLayNet 11 classes ───────────────────────────────────────────
    class_names: list = field(default_factory=lambda: [
        "Caption",        # 0
        "Footnote",       # 1
        "Formula",        # 2
        "List-item",      # 3
        "Page-footer",    # 4
        "Page-header",    # 5
        "Picture",        # 6
        "Section-header", # 7
        "Table",          # 8
        "Text",           # 9
        "Title",          # 10
    ])

    text_classes: tuple    = ("Caption", "Footnote", "List-item",
                               "Page-footer", "Page-header",
                               "Section-header", "Text", "Title")
    vision_classes: tuple  = ("Table", "Formula", "Picture")

    min_area_filter: float = 800.0

    # Per-class conf overrides — classes prone to false positives use higher threshold
    class_conf_override: dict = field(default_factory=lambda: {
        "Picture":        0.50,
        "Section-header": 0.40,
        "Caption":        0.40,
    })

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
print("Config ready:", cfg)


## 2. Data models

In [3]:
from pydantic import BaseModel, Field
from typing import Optional, Any

class BBox(BaseModel):
    x1: float; y1: float; x2: float; y2: float

    @property
    def w(self): return self.x2 - self.x1
    @property
    def h(self): return self.y2 - self.y1
    @property
    def area(self): return self.w * self.h
    @property
    def cx(self): return (self.x1 + self.x2) / 2
    @property
    def cy(self): return (self.y1 + self.y2) / 2

    def iou(self, other: "BBox") -> float:
        ix1 = max(self.x1, other.x1); iy1 = max(self.y1, other.y1)
        ix2 = min(self.x2, other.x2); iy2 = min(self.y2, other.y2)
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        union = self.area + other.area - inter
        return inter / union if union > 0 else 0.0

    def to_xyxy_int(self):
        return int(self.x1), int(self.y1), int(self.x2), int(self.y2)

    def scale(self, sx: float, sy: float) -> "BBox":
        return BBox(x1=self.x1*sx, y1=self.y1*sy,
                    x2=self.x2*sx, y2=self.y2*sy)

class LayoutBlock(BaseModel):
    block_id: int
    class_name: str
    conf: float
    bbox: BBox                       # coordinates in original image space
    reading_order: int = -1
    raw_text: str = ""
    ocr_conf: float = 1.0
    ocr_method: str = "paddle"             # "paddle" | "gemini_vision" | "recovery"
    processed_content: Any = None    # Markdown / LaTeX / parsed dict

class PageOutput(BaseModel):
    page_index: int
    image_path: str
    blocks: list[LayoutBlock] = []
    structured_markdown: str = ""
    structured_json: dict = {}
    page_summary: str = ""
    keywords: list[str] = []
    metadata: dict = {}
    status: Literal["ok", "needs_review", "failed"] = "ok"

print("Data models defined.")

Data models defined.


## 3. Stage 1 — Preprocessing

In [ ]:
import cv2
import numpy as np
from PIL import Image
from pathlib import Path

try:
    from pdf2image import convert_from_path
    HAS_PDF2IMAGE = True
except ImportError:
    HAS_PDF2IMAGE = False
    print("pdf2image not installed — PDF rendering disabled.")


def render_pdf(pdf_path: str, dpi: int) -> list[np.ndarray]:
    """Convert each PDF page to an OpenCV BGR image at given DPI."""
    if not HAS_PDF2IMAGE:
        raise RuntimeError("Install pdf2image + poppler to process PDFs.")
    pil_pages = convert_from_path(pdf_path, dpi=dpi, poppler_path=cfg.poppler_path)
    return [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in pil_pages]


def deskew(img: np.ndarray, max_angle_deg: float = 15.0) -> np.ndarray:
    """Detect and correct skew using Hough line transform."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 0, 255,
                            cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 10:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) > max_angle_deg:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def denoise_and_sharpen(img: np.ndarray) -> np.ndarray:
    """Light denoising + sharpening to improve OCR accuracy."""
    denoised = cv2.fastNlMeansDenoisingColored(img, None, 6, 6, 7, 21)
    kernel = np.array([[0, -0.5, 0], [-0.5, 3, -0.5], [0, -0.5, 0]])
    return cv2.filter2D(denoised, -1, kernel)


def resize_to_gt(img: np.ndarray, target: int = None) -> np.ndarray:
    """Resize image to target×target (DocLayNet GT space = 1025×1025).
    Pads with white to preserve aspect ratio before resizing.
    """
    target = target or cfg.gt_imgsz
    h, w = img.shape[:2]
    scale = target / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    # Pad to exact target×target
    canvas = np.full((target, target, 3), 255, dtype=np.uint8)
    canvas[:new_h, :new_w] = resized
    return canvas


def preprocess_for_yolo(img: np.ndarray) -> np.ndarray:
    """Prepare image for YOLO: deskew + resize to GT space (1025×1025)."""
    img = deskew(img)
    img = resize_to_gt(img)
    return img


def preprocess_for_ocr(img: np.ndarray) -> np.ndarray:
    """Prepare high-res image for OCR: deskew + denoise (no downscale)."""
    img = deskew(img)
    img = denoise_and_sharpen(img)
    return img


def load_input(path: str, dpi: int) -> list[np.ndarray]:
    """Accept PDF, PNG, JPG, TIFF. Returns list of BGR images (1 per page)."""
    p = Path(path)
    if p.suffix.lower() == ".pdf":
        return render_pdf(path, dpi=dpi)
    img = cv2.imread(str(p))
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return [img]

print("Stage 1 (Preprocessing) ready.")


## 4. Stage 2 — Layout Detection (YOLOv11s + bbox cleanup + reading order)

In [5]:
from ultralytics import YOLO
import itertools

# ── Load model (once) ─────────────────────────────────────────────────
_yolo_model: YOLO | None = None

def get_yolo(weights: str = cfg.yolo_weights) -> YOLO:
    global _yolo_model
    if _yolo_model is None:
        _yolo_model = YOLO(weights)
        print(f"YOLO loaded: {weights}")
    return _yolo_model


# ── Detection ─────────────────────────────────────────────────────────
def detect_layout(img: np.ndarray) -> list[LayoutBlock]:
    """Run YOLOv11s and return raw LayoutBlocks."""
    model = get_yolo()
    results = model.predict(
        source=img,
        conf=cfg.yolo_conf,
        iou=cfg.yolo_iou,
        imgsz=cfg.yolo_imgsz,
        max_det=cfg.yolo_max_det,
        device=cfg.yolo_device,
        verbose=False,
    )[0]

    blocks = []
    for i, box in enumerate(results.boxes):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        name   = cfg.class_names[cls_id] if cls_id < len(cfg.class_names) else str(cls_id)
        min_conf = cfg.class_conf_override.get(name, cfg.yolo_conf)
        if conf < min_conf:
            continue
        blocks.append(LayoutBlock(
            block_id=i,
            class_name=name,
            conf=round(conf, 4),
            bbox=BBox(x1=x1, y1=y1, x2=x2, y2=y2),
        ))
    return blocks


# ── Bbox cleanup: NMS + small-block filter ────────────────────────────
def nms_blocks(blocks: list[LayoutBlock],
               iou_threshold: float = 0.5) -> list[LayoutBlock]:
    """Class-agnostic NMS — keep higher-confidence box when IoU > threshold."""
    blocks = sorted(blocks, key=lambda b: b.conf, reverse=True)
    kept = []
    for b in blocks:
        if all(b.bbox.iou(k.bbox) < iou_threshold for k in kept):
            kept.append(b)
    return kept


def filter_small_blocks(blocks: list[LayoutBlock],
                        min_area: float = None) -> list[LayoutBlock]:
    """Remove degenerate detections (very small bboxes)."""
    if min_area is None:
        min_area = cfg.min_area_filter
    return [b for b in blocks if b.bbox.area >= min_area]


# ── Column detection ──────────────────────────────────────────────────
def detect_columns(blocks: list[LayoutBlock],
                   img_width: int) -> int:
    """Heuristic: check if blocks cluster into 2 columns."""
    if not blocks:
        return 1
    mid = img_width / 2
    left  = sum(1 for b in blocks if b.bbox.cx < mid)
    right = sum(1 for b in blocks if b.bbox.cx >= mid)
    if min(left, right) / max(left, right) > 0.35:
        return 2
    return 1


# ── Reading order assignment ──────────────────────────────────────────
def assign_reading_order(blocks: list[LayoutBlock],
                          img_width: int) -> list[LayoutBlock]:
    """Sort blocks by reading order: top-to-bottom, left-to-right per row.
    Handles both single-column and two-column layouts.
    """
    n_cols = detect_columns(blocks, img_width)

    if n_cols == 2:
        mid = img_width / 2
        left_blocks  = [b for b in blocks if b.bbox.cx <  mid]
        right_blocks = [b for b in blocks if b.bbox.cx >= mid]
        left_blocks  = sorted(left_blocks,  key=lambda b: (b.bbox.y1, b.bbox.x1))
        right_blocks = sorted(right_blocks, key=lambda b: (b.bbox.y1, b.bbox.x1))
        ordered = left_blocks + right_blocks
    else:
        sorted_y = sorted(blocks, key=lambda b: b.bbox.y1)
        rows: list[list[LayoutBlock]] = []
        for b in sorted_y:
            placed = False
            for row in rows:
                # Use full y-extent of the row, not just the first block
                row_y1 = min(r.bbox.y1 for r in row)
                row_y2 = max(r.bbox.y2 for r in row)
                if b.bbox.y1 < row_y2 and b.bbox.y2 > row_y1:
                    row.append(b)
                    placed = True
                    break
            if not placed:
                rows.append([b])
        ordered = []
        for row in rows:
            ordered.extend(sorted(row, key=lambda b: b.bbox.x1))

    for i, b in enumerate(ordered):
        b.reading_order = i
    return ordered


def run_layout_detection(img: np.ndarray) -> list[LayoutBlock]:
    """Full layout pipeline: detect → NMS → filter → reading order."""
    blocks = detect_layout(img)
    blocks = nms_blocks(blocks)
    blocks = filter_small_blocks(blocks)
    blocks = assign_reading_order(blocks, img.shape[1])
    return blocks

print(f"Stage 2 (Layout Detection) ready. DocLayNet classes: {cfg.class_names}")

Stage 2 (Layout Detection) ready. DocLayNet classes: ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title']


## 5. Stage 3 — OCR Routing

### 5a. PaddleOCR engine

In [ ]:
# Stage 3a — OCR engine: Doctr (python-doctr)
import os as _os, io as _io
_os.environ["USE_TORCH"] = "1"  # must be set before importing doctr
import torch as _torch
from doctr.models import ocr_predictor as _ocr_predictor
from doctr.io import DocumentFile as _DocumentFile
from PIL import Image as _PILImage

_doctr_model = None

def get_doctr_model():
    global _doctr_model
    if _doctr_model is None:
        _doctr_model = _ocr_predictor(pretrained=True)
        if _torch.cuda.is_available():
            _doctr_model = _doctr_model.cuda()
        print(f"Doctr loaded (gpu={_torch.cuda.is_available()}).")
    return _doctr_model


def _bgr_to_png_bytes(img: np.ndarray) -> bytes:
    pil = _PILImage.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    buf = _io.BytesIO()
    pil.save(buf, format="PNG")
    return buf.getvalue()


def crop_block(img_hires: np.ndarray, bbox, pad: int = 6) -> np.ndarray:
    """Crop bbox from high-res image. Upscale if crop height < cfg.ocr_min_crop_h."""
    h, w = img_hires.shape[:2]
    x1, y1, x2, y2 = bbox.to_xyxy_int()
    x1 = max(0, x1 - pad); y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad); y2 = min(h, y2 + pad)
    crop = img_hires[y1:y2, x1:x2]
    if crop.size == 0:
        return crop
    min_h = cfg.ocr_min_crop_h
    if crop.shape[0] < min_h:
        scale = min_h / crop.shape[0]
        new_w = max(1, int(crop.shape[1] * scale))
        crop = cv2.resize(crop, (new_w, min_h), interpolation=cv2.INTER_CUBIC)
    return crop


def scale_blocks_to_hires(blocks: list, img_yolo: np.ndarray,
                           img_hires: np.ndarray) -> list:
    """Scale bbox coordinates from YOLO-input image space to hires image space."""
    yolo_h, yolo_w = img_yolo.shape[:2]
    hires_h, hires_w = img_hires.shape[:2]
    sx = hires_w / yolo_w
    sy = hires_h / yolo_h
    for b in blocks:
        b.bbox = b.bbox.scale(sx, sy)
    return blocks


def run_ocr(crop: np.ndarray) -> tuple:
    """Run Doctr on a crop. Returns (text, confidence)."""
    if crop is None or crop.size == 0:
        return "", 0.0
    model = get_doctr_model()
    doc = _DocumentFile.from_images([_bgr_to_png_bytes(crop)])
    result = model(doc)
    words = []
    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    if word.value.strip():
                        words.append(word.value)
    text = " ".join(words)
    return text, 1.0 if text else 0.0


print("Stage 3a (Doctr — GPU) ready.")

### 5b. Gemini Vision (Tables, Formulas, Pictures)

In [7]:
import google.genai as genai
import time, io, base64

_genai_client = None

def get_genai_client():
    global _genai_client
    if _genai_client is None:
        _genai_client = genai.Client(api_key=cfg.gemini_api_key)
    return _genai_client


def _cv2_to_pil(img):
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


def _call_gemini(prompt: str, image=None,
                 retries: int = 3, delay: float = 2.0) -> str:
    """Call Gemini with optional image. Retries on transient errors."""
    client = get_genai_client()
    contents = [prompt]
    if image is not None:
        contents.append(_cv2_to_pil(image))
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(
                model=cfg.gemini_model,
                contents=contents,
            )
            return resp.text.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(delay * (attempt + 1))
            else:
                raise RuntimeError(f"Gemini failed after {retries} attempts: {e}")


# ── Gemini prompts ────────────────────────────────────────────────────
TABLE_PROMPT = """\
You are a precise document extraction assistant.
The image shows a table from a document.

Extract the table and return it as a valid JSON object with this structure:
{
  "headers": ["col1", "col2", ...],
  "rows": [["val", "val", ...], ...]
}

Rules:
- Preserve the original text exactly as it appears.
- Use null for empty cells.
- Return ONLY the JSON object, no markdown fences, no explanation.
"""

FORMULA_PROMPT = """\
You are a LaTeX expert.
The image shows a mathematical formula or equation.

Convert the formula to LaTeX. Return ONLY the LaTeX expression, wrapped
in $$ delimiters if it is a display equation, or $ delimiters for inline.
No explanation, no code fences.
"""

PICTURE_PROMPT = """\
You are a document analysis assistant.
Describe the figure, chart, diagram, or image shown.
Be concise (2-4 sentences). Focus on what information it conveys.
Return ONLY the description text.
"""

RECOVERY_PROMPT_TEMPLATE = """\
You are an expert OCR correction assistant.
The following text was extracted by an OCR engine but has low confidence.
It may contain errors, missing words, or garbled characters.

Original OCR output:
---
{raw_text}
---

The image of this text block is also provided.
Correct the text based on the image. Return ONLY the corrected text,
no explanation, no metadata.
"""


def gemini_extract_table(crop) -> str:
    return _call_gemini(TABLE_PROMPT, image=crop)

def gemini_extract_formula(crop) -> str:
    return _call_gemini(FORMULA_PROMPT, image=crop)

def gemini_describe_picture(crop) -> str:
    return _call_gemini(PICTURE_PROMPT, image=crop)

def gemini_recovery(crop, raw_text: str) -> str:
    prompt = RECOVERY_PROMPT_TEMPLATE.format(raw_text=raw_text)
    return _call_gemini(prompt, image=crop)

print("Stage 3b (Gemini Vision) ready.")


Stage 3b (Gemini Vision) ready.


### 5c. OCR router

In [ ]:
def ocr_route_block(block, img_hires: np.ndarray) -> object:
    """Route a single block to the appropriate OCR method.
    img_hires: high-res image — bbox already scaled to this space.
    """
    crop = crop_block(img_hires, block.bbox)

    if block.class_name in cfg.vision_classes:
        if block.class_name == "Table":
            block.raw_text   = gemini_extract_table(crop)
            block.ocr_method = "gemini_vision_table"
        elif block.class_name == "Formula":
            block.raw_text   = gemini_extract_formula(crop)
            block.ocr_method = "gemini_vision_formula"
        else:
            block.raw_text   = gemini_describe_picture(crop)
            block.ocr_method = "gemini_vision_picture"
        block.ocr_conf = 1.0

    elif block.class_name in cfg.text_classes:
        text, conf = run_ocr(crop)
        block.raw_text   = text
        block.ocr_conf   = conf
        block.ocr_method = "doctr"

        if conf < cfg.ocr_conf_threshold and text.strip():
            try:
                corrected        = gemini_recovery(crop, text)
                block.raw_text   = corrected
                block.ocr_method = "doctr+gemini_recovery"
                block.ocr_conf   = 0.9
            except Exception as e:
                print(f"  [WARNING] Recovery failed for block {block.block_id}: {e}")
    else:
        text, conf = run_ocr(crop)
        block.raw_text   = text
        block.ocr_conf   = conf
        block.ocr_method = "doctr_fallback"

    return block


def run_ocr_all_blocks(blocks: list, img_hires: np.ndarray) -> list:
    """Run OCR on every block using high-res image, sorted by reading order."""
    ordered = sorted(blocks, key=lambda b: b.reading_order)
    for i, block in enumerate(ordered):
        print(f"  OCR block {i+1}/{len(ordered)} "
              f"[{block.class_name}] "
              f"conf={block.conf:.2f} "
              f"bbox=({int(block.bbox.x1)},{int(block.bbox.y1)})")
        ocr_route_block(block, img_hires)
    return ordered

print("Stage 3c (OCR Router) ready.")

## 6. Stage 4 — LLM Post-processing

In [9]:
import json, re

# ── Single-block fallback prompt ──────────────────────────────────────
NORMALIZE_TEXT_PROMPT = """\
You are a document formatting assistant.
Below is raw text extracted from a document block of type "{class_name}".

Raw text:
---
{raw_text}
---

Tasks:
1. Fix any OCR errors (character substitutions, broken words, missing spaces).
2. Format as clean Markdown appropriate for the block type:
   - Title / Section-header → `# Heading` or `## Heading`
   - List-item → `- item`
   - Caption → plain paragraph preceded by **Caption:** label
   - Other text types → plain paragraph
3. Preserve all original content — do not add or remove information.

Return ONLY the formatted Markdown, no explanation.
"""

# ── Batch prompt — all text blocks in one Gemini call ─────────────────
BATCH_NORMALIZE_PROMPT = """\
You are a document formatting assistant.
Format each of the following {n} OCR-extracted text blocks as clean Markdown.

Rules per block type:
- Title / Section-header → # or ## heading
- List-item → - bullet item
- Caption → **Caption:** followed by the text
- All other types → plain paragraph

Fix OCR errors. Preserve all content — do not add or remove information.

{blocks_input}

Return the formatted Markdown for each block in the same order.
Separate each block's output with exactly this marker on its own line:
<<<SEP>>>
Output exactly {n} sections in order, nothing else.
"""


def normalize_text_block(block: LayoutBlock) -> LayoutBlock:
    """Fallback: post-process a single text block via Gemini."""
    if not block.raw_text.strip():
        block.processed_content = ""
        return block
    prompt = NORMALIZE_TEXT_PROMPT.format(
        class_name=block.class_name,
        raw_text=block.raw_text,
    )
    block.processed_content = _call_gemini(prompt)
    return block


def batch_normalize_text_blocks(blocks: list[LayoutBlock]) -> None:
    """Normalize all text blocks in a single Gemini call instead of N calls.
    Falls back to individual calls if Gemini returns the wrong number of parts.
    """
    non_empty = [b for b in blocks if b.raw_text.strip()]
    for b in blocks:
        if not b.raw_text.strip():
            b.processed_content = ""

    if not non_empty:
        return

    lines = [f"[Block {i+1} | {b.class_name}]\n{b.raw_text}"
             for i, b in enumerate(non_empty)]
    blocks_input = "\n<<<SEP>>>\n".join(lines)
    prompt = BATCH_NORMALIZE_PROMPT.format(
        n=len(non_empty),
        blocks_input=blocks_input,
    )
    try:
        raw   = _call_gemini(prompt)
        parts = raw.split("<<<SEP>>>")
        if len(parts) == len(non_empty):
            for block, part in zip(non_empty, parts):
                block.processed_content = part.strip()
            return
        print(f"  [WARN] Batch normalize: expected {len(non_empty)} parts, "
              f"got {len(parts)} — falling back to individual calls.")
    except Exception as e:
        print(f"  [WARN] Batch normalize failed ({e}) — falling back.")

    for block in non_empty:
        normalize_text_block(block)


def normalize_table_block(block: LayoutBlock) -> LayoutBlock:
    """Parse Table JSON and convert to Markdown table."""
    raw = block.raw_text.strip()
    raw = re.sub(r"^```[a-z]*\n?", "", raw)
    raw = re.sub(r"```$", "", raw).strip()
    try:
        tbl     = json.loads(raw)
        headers = tbl.get("headers", [])
        rows    = tbl.get("rows", [])
        md  = "| " + " | ".join(str(h) for h in headers) + " |\n"
        md += "| " + " | ".join("---" for _ in headers) + " |\n"
        for row in rows:
            md += "| " + " | ".join(str(c) if c is not None else "" for c in row) + " |\n"
        block.processed_content = md
    except json.JSONDecodeError:
        block.processed_content = raw
    return block


def post_process_blocks(blocks: list[LayoutBlock]) -> list[LayoutBlock]:
    """Apply post-processing to every block.
    Text blocks are batched into one Gemini call; vision blocks handled individually.
    """
    sorted_blocks = sorted(blocks, key=lambda b: b.reading_order)

    text_blocks = [b for b in sorted_blocks
                   if b.class_name not in ("Table", "Formula", "Picture")]
    batch_normalize_text_blocks(text_blocks)

    for block in sorted_blocks:
        if block.class_name == "Table":
            normalize_table_block(block)
        elif block.class_name in ("Formula", "Picture"):
            block.processed_content = block.raw_text

    return blocks

print("Stage 4 (Post-processing) ready.")

Stage 4 (Post-processing) ready.


## 7. Stage 5 — Schema Validation + Retry

In [10]:
# Stage 5 + 6 — Schema validation + Page summary (1 Gemini call)
from pydantic import BaseModel, Field
from typing import Optional

class PageSchema(BaseModel):
    title: Optional[str] = None
    sections: list[dict] = Field(default_factory=list)


def _blocks_to_text(blocks: list) -> str:
    parts = []
    for b in sorted(blocks, key=lambda x: x.reading_order):
        content = b.processed_content or b.raw_text
        parts.append(f"[{b.class_name}] {content}")
    return "\n".join(parts)


COMBINED_SCHEMA_SUMMARY_PROMPT = """\
You are a structured document analysis assistant.
Below are blocks extracted from a single document page (in reading order).

Blocks:
---
{blocks_text}
---

Return a single JSON object with this exact structure:
{{
  \"structured_json\": {{
    \"title\": \"<string | null>\",
    \"sections\": [
      {{
        \"heading\": \"<string | null>\",
        \"paragraphs\": [\"<string>\"],
        \"tables\": [{{\"headers\": [], \"rows\": [[]]}}],
        \"formulas\": [\"<LaTeX>\"],
        \"figures\": [\"<description>\"]
      }}
    ]
  }},
  \"summary\": \"<3-5 sentence summary of the page>\",
  \"keywords\": [\"kw1\", \"kw2\"],
  \"page_type\": \"<title_page|table_of_contents|body|figure_page|reference|other>\",
  \"language\": \"<ISO 639-1>\"
}}

Return ONLY valid JSON. No markdown fences. No explanation.\
"""


def assemble_and_summarize(blocks: list, max_retries: int = None) -> tuple:
    """Assemble structured JSON and generate page summary in a single Gemini call.
    Returns (structured_json, is_valid, summary, keywords, metadata).
    """
    max_retries = max_retries or cfg.max_retries
    blocks_text = _blocks_to_text(blocks)

    if not blocks_text.strip():
        return {}, True, "Empty page.", [], {}

    prompt = COMBINED_SCHEMA_SUMMARY_PROMPT.format(blocks_text=blocks_text)
    for attempt in range(max_retries):
        try:
            raw = _call_gemini(prompt)
            raw = re.sub(r"^```[a-z]*\n?", "", raw)
            raw = re.sub(r"```$", "", raw).strip()
            data = json.loads(raw)
            structured = data.get("structured_json", {})
            PageSchema(**structured)
            summary  = data.get("summary", "")
            keywords = data.get("keywords", [])
            meta     = {k: data[k] for k in ("summary", "keywords", "page_type", "language") if k in data}
            return structured, True, summary, keywords, meta
        except Exception as e:
            print(f"  [Schema+Summary] Attempt {attempt+1}/{max_retries} failed: {e}")

    print("  [Schema+Summary] All attempts failed — returning empty.")
    return {"error": "parse_failed"}, False, "", [], {}


print("Stages 5 & 6 ready (combined call).")

Stage 5 (Schema Validation) ready.


## 8. Stage 6 — Page Summary

In [11]:
# Stage 6 merged into Stage 5 above
print('Stage 6 merged.')

Stage 6 (Page Summary) ready.


## 9. Full pipeline orchestrator

In [ ]:
def process_page(img_bgr: np.ndarray,
                 page_index: int,
                 img_hires_bgr: np.ndarray = None,
                 image_path: str = "") -> object:
    """Run the full 6-stage pipeline on a single page.

    img_bgr       : image rendered at pdf_dpi — used for YOLO layout detection
    img_hires_bgr : image rendered at ocr_dpi — used for OCR crop (if None = img_bgr)
    """
    print(f"\n{'='*60}")
    print(f"Processing page {page_index}")
    print(f"{'='*60}")

    out = PageOutput(page_index=page_index, image_path=image_path)

    # Stage 1 — Preprocessing
    print("[1/6] Preprocessing...")
    # img_yolo: resize+pad to 1025×1025 (GT space) for YOLO and OCR crop
    img_yolo  = preprocess_for_yolo(img_bgr)
    # img_hires: deskew+denoise only, keep high resolution — NOT used for cropping
    # (bbox is in 1025×1025 space, scaling to hires shifts due to padding)
    if img_hires_bgr is not None:
        preprocess_for_ocr(img_hires_bgr)  # run for warm-up but not used for crop
    print(f"  YOLO/OCR img : {img_yolo.shape[1]}×{img_yolo.shape[0]} px (GT space)")

    # Stage 2 — Layout detection
    print("[2/6] Layout detection...")
    blocks = run_layout_detection(img_yolo)
    print(f"  → {len(blocks)} blocks detected")

    out.blocks = blocks

    if not blocks:
        out.status = "needs_review"
        print("  [WARNING] No blocks detected — marking needs_review.")
        return out

    # Stage 3 — OCR routing (crop from img_yolo — bbox is already in this space)
    print("[3/6] OCR routing...")
    blocks = run_ocr_all_blocks(blocks, img_yolo)

    # Stage 4 — Post-processing
    print("[4/6] LLM post-processing...")
    blocks = post_process_blocks(blocks)

    out.structured_markdown = "\n\n".join(
        str(b.processed_content or b.raw_text)
        for b in sorted(blocks, key=lambda b: b.reading_order)
        if (b.processed_content or b.raw_text).strip()
    )

    # Stage 5 — Schema validation
    print("[5+6/6] Schema + Summary (1 call)...")
    json_data, is_valid, summary, keywords, meta = assemble_and_summarize(blocks)
    out.structured_json = json_data
    if not is_valid:
        out.status = "needs_review"
    out.page_summary = summary
    out.keywords     = keywords
    out.metadata     = meta

    print(f"\n✓ Page {page_index} done. Status: {out.status}")
    print(f"  Summary: {summary[:120]}..." if len(summary) > 120 else f"  Summary: {summary}")
    print(f"  Keywords: {keywords}")
    return out


def process_document(input_path: str) -> list:
    """Process a full document (PDF or image). Returns one PageOutput per page."""
    print(f"\nLoading document: {input_path}")

    pages_yolo  = load_input(input_path, dpi=cfg.pdf_dpi)
    pages_hires = load_input(input_path, dpi=cfg.ocr_dpi)
    print(f"→ {len(pages_yolo)} page(s) found.")
    print(f"  YOLO DPI={cfg.pdf_dpi}  OCR DPI={cfg.ocr_dpi}")

    results = []
    for i, (page_yolo, page_hires) in enumerate(zip(pages_yolo, pages_hires)):
        page_out = process_page(
            page_yolo,
            page_index=i,
            img_hires_bgr=page_hires,
            image_path=f"{input_path}::page{i}",
        )
        results.append(page_out)

        out_path = cfg.output_dir / f"page_{i:03d}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(page_out.model_dump_json(indent=2))
        print(f"  Saved → {out_path}")

    return results

print("Pipeline orchestrator ready.")


## 10. Visualisation helpers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CLASS_COLORS = {
    "Caption":        "#4E79A7",
    "Footnote":       "#59A14F",
    "Formula":        "#F28E2B",
    "List-item":      "#76B7B2",
    "Page-footer":    "#BAB0AC",
    "Page-header":    "#BAB0AC",
    "Picture":        "#E15759",
    "Section-header": "#FF9DA7",
    "Table":          "#9C755F",
    "Text":           "#4E79A7",
    "Title":          "#EDC948",
}


def visualise_layout(img: np.ndarray,
                     blocks: list[LayoutBlock],
                     title: str = "Layout Detection",
                     dpi: int = 150) -> None:
    """Draw bboxes with class labels and reading-order numbers."""
    h, w = img.shape[:2]
    fig_w = 18
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h), dpi=dpi)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    for b in blocks:
        color = CLASS_COLORS.get(b.class_name, "#AAAAAA")
        x1, y1, x2, y2 = b.bbox.to_xyxy_int()
        rect = mpatches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            boxstyle="square,pad=0",
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4,
                f"{b.reading_order}: {b.class_name} ({b.conf:.2f})",
                color=color, fontsize=7, fontweight="bold",
                bbox=dict(facecolor="white", alpha=0.6, pad=1, edgecolor="none"))

    legend_patches = [
        mpatches.Patch(color=c, label=n)
        for n, c in CLASS_COLORS.items()
        if any(b.class_name == n for b in blocks)
    ]
    ax.legend(handles=legend_patches, loc="upper right", fontsize=8)
    ax.set_title(title, fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def print_page_report(page_out: PageOutput) -> None:
    print(f"\n{'━'*60}")
    print(f"Page {page_out.page_index}  |  Status: {page_out.status}")
    print(f"Blocks: {len(page_out.blocks)}")
    print(f"Summary: {page_out.page_summary}")
    print(f"Keywords: {', '.join(page_out.keywords)}")
    print(f"\nMarkdown preview (first 500 chars):")
    print(page_out.structured_markdown[:500])
    print(f"{'━'*60}")

print("Visualisation helpers ready.")


## 11. Run — change INPUT_PATH to your file

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  EDIT THESE:
INPUT_PATH = r"C:\Users\HP\Desktop\Kubbies\KhoaLuan\GT\staging\sample_0984_2c0abc344218b9abe553863f4bf8a938817a5b3de9b8d79ad840bb07b76bc9a9.pdf"
cfg.yolo_weights   = r"C:\Users\HP\Desktop\Kubbies\KhoaLuan\dataset\BaseC\best.pt"
cfg.gemini_api_key = os.getenv("GEMINI_API_KEY", "")
cfg.poppler_path   = r"C:\Users\HP\Desktop\Kubbies\KhoaLuan\poppler\Library\bin"
# ──────────────────────────────────────────────────────────────────────

_genai_client = None

from pathlib import Path as _Path
if not _Path(INPUT_PATH).exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

results = process_document(INPUT_PATH)

for page_out in results:
    print_page_report(page_out)

# Visualise layout — use 300dpi image for clearer display
if results:
    pages_hires = load_input(INPUT_PATH, dpi=cfg.ocr_dpi)   # 300 dpi
    first_img   = preprocess_for_yolo(pages_hires[0])
    visualise_layout(first_img, results[0].blocks,
                     title=f"Layout — page 0 ({Path(INPUT_PATH).name})")


## 12. Save combined output
Merge all pages into a single Markdown file and a combined JSON.

In [ ]:
import math

def _bbox_xyxy_to_coco(x1, y1, x2, y2):
    """Convert xyxy absolute coords to COCO [x, y, w, h]."""
    return [round(x1, 4), round(y1, 4), round(x2 - x1, 4), round(y2 - y1, 4)]


def _block_to_text(block) -> str:
    content = block.processed_content or block.raw_text or ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, dict):
        headers = content.get("headers", [])
        rows = content.get("rows", [])
        if headers or rows:
            md = "| " + " | ".join(str(h) for h in headers) + " |\n"
            md += "| " + " | ".join("---" for _ in headers) + " |\n"
            for row in rows:
                md += "| " + " | ".join(str(c) if c is not None else "" for c in row) + " |\n"
            return md.strip()
    return str(content).strip()


LABEL_META = {
    "Caption":        {"category_id": 1,  "yolo_class_id": 0,  "content_role": "caption",        "is_relevant": True},
    "Footnote":       {"category_id": 2,  "yolo_class_id": 1,  "content_role": "footnote",       "is_relevant": True},
    "Formula":        {"category_id": 3,  "yolo_class_id": 2,  "content_role": "formula",        "is_relevant": True},
    "List-item":      {"category_id": 4,  "yolo_class_id": 3,  "content_role": "list_item",      "is_relevant": True},
    "Page-footer":    {"category_id": 5,  "yolo_class_id": 4,  "content_role": "footer",         "is_relevant": False},
    "Page-header":    {"category_id": 5,  "yolo_class_id": 5,  "content_role": "header",         "is_relevant": False},
    "Picture":        {"category_id": 7,  "yolo_class_id": 6,  "content_role": "picture",        "is_relevant": True},
    "Section-header": {"category_id": 6,  "yolo_class_id": 7,  "content_role": "section_header", "is_relevant": True},
    "Table":          {"category_id": 9,  "yolo_class_id": 8,  "content_role": "table",          "is_relevant": True},
    "Text":           {"category_id": 10, "yolo_class_id": 9,  "content_role": "text",           "is_relevant": True},
    "Title":          {"category_id": 11, "yolo_class_id": 10, "content_role": "title",          "is_relevant": True},
}


def results_to_gt_format(results: list, input_path: str) -> dict:
    """Convert pipeline PageOutput list to GT v2.0 format.
    Bbox coordinates are in 1025×1025 space (same as DocLayNet GT).
    """
    from pathlib import Path as _P
    p = _P(input_path)
    doc_id = p.stem

    all_pages = []
    total_annotations = 0
    label_dist: dict = {}

    for page_out in results:
        layout_gt = []
        for ann_id, block in enumerate(
            sorted(page_out.blocks, key=lambda b: b.reading_order)
        ):
            meta = LABEL_META.get(block.class_name, {
                "category_id": 0, "yolo_class_id": -1,
                "content_role": "unknown", "is_relevant": True,
            })
            bbox_coco = _bbox_xyxy_to_coco(
                block.bbox.x1, block.bbox.y1,
                block.bbox.x2, block.bbox.y2,
            )
            text = _block_to_text(block)
            is_vision = block.class_name in ("Picture", "Formula")
            layout_gt.append({
                "annotation_id":              ann_id,
                "category_id":                meta["category_id"],
                "yolo_class_id":              meta["yolo_class_id"],
                "label_name":                 block.class_name,
                "bbox_coco":                  bbox_coco,
                "reading_order":              block.reading_order + 1,
                "content_summary":            "",
                "content_role":               meta["content_role"],
                "is_relevant_for_extraction": meta["is_relevant"],
                "ocr_confidence":             "pipeline",
                "needs_review":               is_vision,
                "review_reason":              "formula_or_picture" if is_vision else "",
                "text":                       text,
            })
            label_dist[block.class_name] = label_dist.get(block.class_name, 0) + 1

        total_annotations += len(layout_gt)
        all_pages.append({
            "page_no":   page_out.page_index,
            "layout_gt": layout_gt,
        })

    meta_page = results[-1].metadata if results else {}
    return {
        "gt_version":          "2.0-pipeline",
        "created_by":          "document_pipeline",
        "created_at":          "",
        "needs_human_review":  any(p.status != "ok" for p in results),
        "doc_id":              doc_id,
        "source": {
            "file_name":     p.name,
            "doc_name":      p.name,
            "page_no":       results[0].page_index if results else 0,
            "collection":    "",
            "doc_category":  meta_page.get("document_type", ""),
            "image_size":    [cfg.gt_imgsz, cfg.gt_imgsz],  # 1025×1025
        },
        "analysis": {
            "document_type":   meta_page.get("document_type", ""),
            "layout_type":     meta_page.get("page_type", ""),
            "language":        meta_page.get("language", ""),
            "quality":         "pipeline",
            "content_density": "",
        },
        "pages":               all_pages,
        "label_distribution":  label_dist,
        "total_annotations":   total_annotations,
    }


def save_gt_output(results: list, input_path: str) -> None:
    from pathlib import Path as _P
    stem = _P(input_path).stem
    out_path = cfg.output_dir / f"{stem}.json"
    data = results_to_gt_format(results, input_path)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"GT-format JSON saved -> {out_path}")

    md_path = cfg.output_dir / f"{stem}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        for p in results:
            f.write(f"\n---\n*Page {p.page_index}*\n\n")
            f.write(p.structured_markdown)
            f.write("\n")
    print(f"Markdown saved -> {md_path}")


save_gt_output(results, INPUT_PATH)


## 13. Batch test — 5 samples

In [ ]:
# -- 13. Batch test runner -- 5 samples
import traceback

STAGING_DIR = r'C:\Users\HP\Desktop\Kubbies\KhoaLuan\GT\staging'

TEST_SAMPLES = [
    r'sample_0181_766a31624c79885e840ff3a6934d1ff2aa13821a660706b0d4cd9509304fc79e.pdf',
    r'sample_0388_36dcbf2f22538d58f32e13947d855f412710d2580f7eed8b76b0c570467fbc53.pdf',
    r'sample_0483_96a32a39c30d621b278f91d0c957521aaaf8faba80c8d21dd13411a2f6f8dc6a.pdf',
    r'sample_0507_c52c3b20f10542c5ca263781401605ddb99f4363e1c551470ae0d471ee9f35ff.pdf',
    r'sample_0593_8ad7a47d65e0e0dbe31e731b864ca88aa9e94f12859e5fbb1e5ffb35c811e1ef.pdf',
]

_batch_results = {}  # stem -> list[PageOutput]

for fname in TEST_SAMPLES:
    pdf_path = str(Path(STAGING_DIR) / fname)
    stem = Path(fname).stem
    print(f'\n{"#"*70}')
    print(f'SAMPLE: {stem[:50]}')
    print(f'{"#"*70}')
    try:
        res = process_document(pdf_path)
        save_gt_output(res, pdf_path)
        _batch_results[stem] = res
        n = sum(len(r.blocks) for r in res)
        print(f'  OK -- {len(res)} page(s), {n} total blocks')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        _batch_results[stem] = []

print('\n' + '='*70)
print('BATCH COMPLETE')
for stem, res in _batch_results.items():
    n_blocks = sum(len(r.blocks) for r in res)
    status = res[0].status if res else 'FAILED'
    print(f'  {stem[:52]:52s}  blocks={n_blocks:3d}  status={status}')
print('='*70)


## 14. Evaluation — pipeline vs GT

In [ ]:
# -- 14. Evaluation -- compare pipeline output vs GT
import json as _json
from pathlib import Path as _P

GT_DIR     = r'C:\Users\HP\Desktop\Kubbies\KhoaLuan\GT\staging'
OUTPUT_DIR = r'C:\Users\HP\Desktop\Kubbies\KhoaLuan\output'

SAMPLE_STEMS = [
    'sample_0181_766a31624c79885e840ff3a6934d1ff2aa13821a660706b0d4cd9509304fc79e',
    'sample_0388_36dcbf2f22538d58f32e13947d855f412710d2580f7eed8b76b0c570467fbc53',
    'sample_0483_96a32a39c30d621b278f91d0c957521aaaf8faba80c8d21dd13411a2f6f8dc6a',
    'sample_0507_c52c3b20f10542c5ca263781401605ddb99f4363e1c551470ae0d471ee9f35ff',
    'sample_0593_8ad7a47d65e0e0dbe31e731b864ca88aa9e94f12859e5fbb1e5ffb35c811e1ef',
]

TEXT_CLASSES_EVAL = {'Caption','Footnote','List-item','Page-footer',
                     'Page-header','Section-header','Text','Title'}

def _load_json(path):
    with open(path, encoding='utf-8') as f:
        return _json.load(f)

print(f'\n{"="*80}')
print(f'{"EVALUATION REPORT":^80}')
print(f'{"="*80}')

total_gt = 0; total_pipe = 0
total_text_with = 0; total_text_all = 0
total_bbox_issues = 0

for stem in SAMPLE_STEMS:
    gt_path  = _P(GT_DIR)  / f'{stem}_v12_merged.json'
    out_path = _P(OUTPUT_DIR) / f'{stem}.json'

    if not gt_path.exists():
        print(f'\n[SKIP] GT not found: {gt_path.name}')
        continue
    if not out_path.exists():
        print(f'\n[SKIP] Output not found: {out_path.name}')
        continue

    gt   = _load_json(gt_path)
    pipe = _load_json(out_path)

    gt_blocks   = gt['total_annotations']
    pipe_blocks = pipe['total_annotations']
    delta_pct   = (pipe_blocks - gt_blocks) / max(gt_blocks, 1) * 100

    gt_dist   = gt.get('label_distribution', {})
    pipe_dist = pipe.get('label_distribution', {})

    pipe_anns = []
    for page in pipe.get('pages', []):
        pipe_anns.extend(page.get('layout_gt', []))

    text_blocks = [a for a in pipe_anns if a['label_name'] in TEXT_CLASSES_EVAL]
    text_with   = [a for a in text_blocks if a.get('text', '').strip()]
    text_cov    = len(text_with) / max(len(text_blocks), 1) * 100

    bbox_bad = []
    for a in pipe_anns:
        x, y, w, h = a.get('bbox_coco', [0,0,0,0])
        if x < 0 or y < 0 or x+w > 1030 or y+h > 1030:
            bbox_bad.append(a['annotation_id'])

    total_gt     += gt_blocks
    total_pipe   += pipe_blocks
    total_text_with += len(text_with)
    total_text_all  += len(text_blocks)
    total_bbox_issues += len(bbox_bad)

    ok_cnt  = 'OK' if abs(delta_pct) <= 30 else 'WARN'
    ok_txt  = 'OK' if text_cov >= 80   else 'WARN'
    ok_bbox = 'OK' if not bbox_bad     else 'WARN'

    print(f'\n{stem[7:15]}...')
    print(f'  Blocks  : GT={gt_blocks:3d}  Pipeline={pipe_blocks:3d}  diff={delta_pct:+.0f}%  [{ok_cnt}]')
    print(f'  TextCov : {len(text_with)}/{len(text_blocks)}  ({text_cov:.0f}%)  [{ok_txt}]')
    print(f'  Bbox    : [{ok_bbox}]' + (f'  bad_ids={bbox_bad}' if bbox_bad else ''))
    print(f'  GT dist : {dict(sorted(gt_dist.items()))}')
    print(f'  Pipe    : {dict(sorted(pipe_dist.items()))}')

print(f'\n{"─"*80}')
print(f'TOTAL  GT={total_gt}  Pipe={total_pipe}  '
      f'TextCov={total_text_with}/{total_text_all} ({total_text_with/max(total_text_all,1)*100:.0f}%)  '
      f'BboxIssues={total_bbox_issues}')
print(f'{"="*80}')
